In [1]:
!pip install -q transformers datasets accelerate torch scikit-learn


In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score


In [4]:
import pandas as pd


In [5]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

train_df.head()


,sentence,label
0,"operating result showed a loss of eur 2.9 mn ,...",0
1,18 march 2010 a leakage in the gypsum pond was...,0
2,finland-based stockmann group has closed seven...,0
3,the company plans to close two of the three li...,0
4,finnish suominen corporation that makes wipes ...,0


In [7]:
from datasets import Dataset


In [8]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)


In [9]:
model_name = "yiyanghkust/finbert-tone"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

In [10]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


In [11]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)


Map:   0%|          | 0/1449 [00:00<?, ? examples/s]

Map:   0%|          | 0/484 [00:00<?, ? examples/s]

Map:   0%|          | 0/484 [00:00<?, ? examples/s]

In [12]:
columns = ["input_ids", "attention_mask", "label"]

train_dataset.set_format(type="torch", columns=columns)
val_dataset.set_format(type="torch", columns=columns)
test_dataset.set_format(type="torch", columns=columns)


In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}


In [31]:
training_args = TrainingArguments(
    output_dir="./finbert_csv",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=2,
    report_to="none"
)


In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-3012789700.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [33]:
trainer.train()


Step,Training Loss
50,0.010900
100,0.015500
150,0.003100
200,0.018000
250,0.000500


TrainOutput(global_step=273, training_loss=0.008930273531448273, metrics={'train_runtime': 97.6971, 'train_samples_per_second': 44.495, 'train_steps_per_second': 2.794, 'total_flos': 285938506715904.0, 'train_loss': 0.008930273531448273, 'epoch': 3.0})

In [34]:
test_results = trainer.evaluate(test_dataset)
test_results


{'eval_loss': 1.3189666271209717,
 'eval_accuracy': 0.7954545454545454,
 'eval_runtime': 3.3094,
 'eval_samples_per_second': 146.248,
 'eval_steps_per_second': 9.367,
 'epoch': 3.0}

In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

albert_name = "albert-base-v2"

albert_tokenizer = AutoTokenizer.from_pretrained(albert_name)

albert_model = AutoModelForSequenceClassification.from_pretrained(
    albert_name,
    num_labels=3
)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
def tokenize_albert(batch):
    return albert_tokenizer(
        batch["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

albert_train = train_dataset.map(tokenize_albert, batched=True)
albert_val = val_dataset.map(tokenize_albert, batched=True)
albert_test = test_dataset.map(tokenize_albert, batched=True)


Map:   0%|          | 0/1449 [00:00<?, ? examples/s]

Map:   0%|          | 0/484 [00:00<?, ? examples/s]

Map:   0%|          | 0/484 [00:00<?, ? examples/s]

In [22]:
columns = ["input_ids", "attention_mask", "label"]

albert_train.set_format("torch", columns=columns)
albert_val.set_format("torch", columns=columns)
albert_test.set_format("torch", columns=columns)


In [23]:
from transformers import TrainingArguments

albert_args = TrainingArguments(
    output_dir="./albert_results",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=2,
    report_to="none"
)


In [24]:
from transformers import Trainer

albert_trainer = Trainer(
    model=albert_model,
    args=albert_args,
    train_dataset=albert_train,
    eval_dataset=albert_val,
    tokenizer=albert_tokenizer,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-2342414559.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  albert_trainer = Trainer(


In [25]:
albert_trainer.train()


Step,Training Loss
50,0.845500
100,0.443300
150,0.262200


TrainOutput(global_step=184, training_loss=0.46131956577301025, metrics={'train_runtime': 116.9413, 'train_samples_per_second': 49.563, 'train_steps_per_second': 1.573, 'total_flos': 34631772521472.0, 'train_loss': 0.46131956577301025, 'epoch': 4.0})

In [26]:
albert_preds = albert_trainer.predict(albert_test)

albert_accuracy = accuracy_score(
    albert_preds.label_ids,
    np.argmax(albert_preds.predictions, axis=1)
)

albert_accuracy


0.8161157024793388